# Continuum — Interactive Recovery Demo

Drives the kill-and-recover sequence against a running Continuum API and shows the
CockroachDB memory layer doing the work — **without cloning or installing anything**
if you point `BASE_URL` at a running instance.

> The claim under test: *the agent's execution environment is allowed to die mid-incident;
> its memory is not.*

**What you need:** a reachable Continuum API. Either
- local — `make run-api` (then `BASE_URL = "http://localhost:8000"`), or
- your own deployment.

The kill beat (sections 4–5) requires a **local** API, because it kills the process by port.
Against a remote URL, run sections 1–3 only.

Recording script for the same flow: [`submission/DEMO_SCRIPT.md`](../submission/DEMO_SCRIPT.md).
Setup notes: [`notebooks/README.md`](README.md).

In [ ]:
import json
import uuid

import httpx

BASE_URL = "http://localhost:8000"  # or your deployment

# One correlation id per notebook session. This is the field the orchestrator
# recovers on: successive alerts carrying the SAME id resume the SAME incident,
# which is what makes section 3 an *advance* rather than a second incident.
# Generated per session so re-running the notebook starts clean instead of
# resuming whatever the last run left behind.
CORRELATION_ID = f"notebook-{uuid.uuid4().hex[:8]}"


def show(obj):
    print(json.dumps(obj, indent=2, default=str))


r = httpx.get(f"{BASE_URL}/api/v1/health", timeout=10)
r.raise_for_status()
show(r.json())
print()
print(f"correlation_id for this session: {CORRELATION_ID}")

## 1. Fire a synthetic alert

All incident data is synthetic (ADR 005). The orchestrator's **first** action on every
invocation is a CockroachDB read for existing open incident state — never a warm cache.

In [ ]:
# This body mirrors DEMO_ALERT in scripts/demo_run.py byte for byte apart from
# `correlation_id`. That matters: sections 4 and 6 advance this incident using
# that script, and the orchestrator re-proposes from the alert text on every
# tick — so a different body here would mean the incident is opened by one alert
# and resumed by another. Keep the two in sync if either changes.
#
# `correlation_id` is REQUIRED by the Alert model in api/main.py — without it
# the API answers 422 and never reaches CockroachDB.
alert = {
    "correlation_id": CORRELATION_ID,
    "service": "checkout-api",
    "region": "us-east-1",
    "severity": "high",
    "text": (
        "[FIRING:1] HighLatencyP99 service=checkout-api region=us-east-1 severity=high — "
        "histogram_quantile(0.99, http_server_duration_seconds) = 2.41s exceeds SLO 0.80s for 5m; "
        "db_pool_connections_active 200/200, db_pool_clients_waiting 47"
    ),
}

r = httpx.post(f"{BASE_URL}/api/v1/alert", json=alert, timeout=60)
r.raise_for_status()
result = r.json()
INCIDENT_ID = result["incident_id"]
show(result)

## 2. Read the live state back over MCP

`/api/v1/incidents/open` is answered by the **application** calling the CockroachDB Cloud
Managed MCP Server's read-only SQL tool — not by a developer in an IDE (ADR 003).

A `503` here is a legitimate, deliberate response: the endpoint surfaces MCP failure rather
than masking it.

In [ ]:
r = httpx.get(f"{BASE_URL}/api/v1/incidents/open", timeout=30)
print(f"HTTP {r.status_code}")
show(r.json())

## 3. Advance the incident one step

Each step commits in **two** explicit `SERIALIZABLE` transactions — `executing` before the
execution window, `executed` after (ADR 009). The gap between them is where a crash lands.

In [ ]:
# Same alert, same correlation_id — so this RESUMES the incident opened above
# rather than opening a second one. Watch `step_index` advance while
# `incident_id` stays identical.
r = httpx.post(f"{BASE_URL}/api/v1/alert", json=alert, timeout=60)
r.raise_for_status()
result = r.json()
show(result)
print()
print(f"same incident as section 1? {result['incident_id'] == INCIDENT_ID}")

## 4. The kill — local only

**This is the point of the project.** A real `SIGKILL` / `TerminateProcess`: no graceful
shutdown, no checkpoint call, no cleanup hook. Fire it *during* a step's execution window
so the step is durably stuck in `executing`.

Run this from a shell next to a live API, in two terminals. Pass `--correlation-id` with the
value printed in cell 1 — **the same incident this notebook opened in section 1** — otherwise
`demo_run.py` drives its own separate incident, and section 5 below (scoped to `INCIDENT_ID`)
would never show a kill that landed on a different row:

```bash
python scripts/demo_run.py --tick --via-api --correlation-id <CORRELATION_ID printed in cell 1>
python scripts/chaos_kill.py --port 8000      # kill it mid-step (terminal 2)
```

PowerShell is identical — both are plain `python` invocations. To start the API without
`make`: `python -m uvicorn api.main:app --port 8000`.

`--via-api` is required: a bare `--tick` runs in-process and finishes before you could kill it.
Time the second command to land inside the step's execution window
(`step_execution_seconds` in `config.py`, 5s by default) but after the Bedrock correlation call
and the `executing` checkpoint commit — too early and nothing durable was written yet.

## 5. Confirm the state outlived the process

With the API dead, query CockroachDB directly. A row sitting in `executing` with no process
alive to own it **is** the thesis — this is the shot the demo video leads with.

Needs `COCKROACH_DATABASE_URL` in the environment.

In [ ]:
import os

import psycopg

# Columns are `status` and `created_at` — see infra/schema.sql. Scoped to this
# session's incident so the output is this run's story, not the whole table.
with psycopg.connect(os.environ["COCKROACH_DATABASE_URL"]) as conn:
    rows = conn.execute(
        """
        SELECT step_index, action, status, created_at
        FROM remediation_steps
        WHERE incident_id = %s
        ORDER BY step_index
        """,
        (INCIDENT_ID,),
    ).fetchall()

print(f"incident {INCIDENT_ID}")
print()
for step_index, action, status, created_at in rows:
    marker = "  <-- died here" if status == "executing" else ""
    stamp = created_at.strftime("%H:%M:%S")
    print(f"{stamp}  step {step_index}  {status:<10} {action}{marker}")

## 6. The recovery

Restart the API and tick again — or, with the orchestrator deployed, let a genuinely cold
Lambda invocation do it. Both need the same `--correlation-id` from cell 1 so this resumes
the incident that was just killed, instead of starting or advancing an unrelated one:

```bash
python -m uvicorn api.main:app --port 8000          # cold restart
python scripts/demo_run.py --tick --via-api --resume-check --correlation-id <CORRELATION_ID from cell 1>
```

Or against the deployed function:

```powershell
# --via-lambda needs an identity with lambda:InvokeFunction. The default identity in
# ~/.aws/credentials is Bedrock-invoke-only and fails with AccessDeniedException; boto3
# also ranks static env keys ABOVE a named profile, so clear them first.
Remove-Item Env:AWS_ACCESS_KEY_ID, Env:AWS_SECRET_ACCESS_KEY -ErrorAction SilentlyContinue
$env:AWS_PROFILE = "continuum-admin"
python scripts/demo_run.py --tick --via-lambda --correlation-id <CORRELATION_ID from cell 1>
```

The interrupted step is **re-executed, not skipped and not duplicated** — then committed
`executed` before the next step is claimed. Re-run cell 5 to watch the frozen row advance.

**The response's `action` may differ from the durable row's, and the row keeps the original.**
The resume path (`resuming=True` in `checkpoint_step_start`) updates `status` and merges
`detail`, but deliberately never rewrites `action` — the durable record keeps the action the
killed invocation committed, while the response reports what *this* pass re-proposed. Claude's
re-proposal is not deterministic, so the two sometimes agree and sometimes don't; neither case
is a bug. What proves exactly-once is the **row count staying put** — re-run cell 5 and count.

The exactly-once property under concurrent invocations is asserted against a real cluster in
[`tests/integration/test_recovery_e2e.py`](../tests/integration/test_recovery_e2e.py), and the
literal process-kill in
[`tests/integration/test_chaos_kill_e2e.py`](../tests/integration/test_chaos_kill_e2e.py).